In [1]:
from pathlib import Path
import pandas as pd

workspace_root = (Path.cwd() / '..').resolve()

# Target JSON (for this notebook context)
json_path = workspace_root / 'JSON Whole Model' / 'Ifc2x3_Duplex_Architecture.json'
assert json_path.exists(), f'JSON not found: {json_path}'

# Read Excel from COBie folder (columns G and I)
excel_path = workspace_root / 'COBie' / 'Uniclass2015_EF_v1_16.xlsx'
assert excel_path.exists(), f'Excel not found: {excel_path}'

# Read only columns G and I
df = pd.read_excel(excel_path, usecols='G,I')
df.columns = ['G', 'I']

# Find rows where Column G == 'Walls'
walls_rows = df[df['G'].astype(str).str.strip().str.casefold() == 'walls']

if walls_rows.empty:
    print("No match found for 'Walls' in column G.")
else:
    first_value = walls_rows['I'].iloc[0]
    print('First corresponding value in column I for Walls:')
    print(first_value)

First corresponding value in column I for Walls:
EF_25_10 : Walls


In [2]:
import json
import pandas as pd

# Load JSON Whole Model data
with json_path.open('r', encoding='utf-8') as f:
    model_data = json.load(f)

def find_wall_property_matches(properties):
    matches = []
    if not isinstance(properties, list):
        return matches

    for prop in properties:
        if not isinstance(prop, dict):
            continue
        haystack_parts = [
            str(prop.get('category', '')),
            str(prop.get('displayName', '')),
            str(prop.get('value', ''))
        ]
        haystack = ' | '.join(haystack_parts).lower()
        if 'wall' in haystack:
            matches.append(prop)
    return matches

result_rows = []
for item in model_data:
    if not isinstance(item, dict):
        continue

    properties = item.get('Properties')
    wall_matches = find_wall_property_matches(properties)
    if not wall_matches:
        continue

    result_rows.append({
        'Name': item.get('Name'),
        'Dbid': item.get('DbId'),
        'WallPropertyMatch': len(wall_matches)
    })

result_table = pd.DataFrame(result_rows)

if result_table.empty:
    print("No objects found with 'Wall' in Properties.")
else:
    result_table['Object Count'] = result_table.groupby('Name')['Name'].transform('count')
    result_table = (
        result_table[['Name', 'Dbid', 'Object Count', 'WallPropertyMatch']]
        .sort_values(['Name', 'Dbid'], kind='stable')
        .reset_index(drop=True)
    )

    print(f"Total rows in result table: {len(result_table)}")
    print("Showing up to 200 rows:")

    display_table = result_table.head(200).reset_index(drop=True)
    with pd.option_context('display.max_rows', 200, 'display.min_rows', 200):
        display(display_table)

Total rows in result table: 231
Showing up to 200 rows:


,Name,Dbid,Object Count,WallPropertyMatch
0,A102,9,1,2
1,A103,10,1,2
2,A104,11,1,2
3,A202,677,1,2
4,A203,676,1,2
5,A204,675,1,2
6,B102,13,1,2
7,B103,14,1,2
8,B202,681,1,2
9,B203,680,1,2
